# Importações


In [1]:
import random
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.svm import SVC

SEEDS = np.arange(1, 11) * 1000

print(SEEDS)

df = pd.read_csv("../datasets/ransomware_transformado.csv", sep=";", decimal=".")

X = df.drop(columns="Benign")
Y = df["Benign"]

df.head()


[ 1000  2000  3000  4000  5000  6000  7000  8000  9000 10000]


,Machine,DebugSize,DebugRVA,MajorImageVersion,MajorOSVersion,ExportRVA,ExportSize,IatVRA,MajorLinkerVersion,MinorLinkerVersion,NumberOfSections,SizeOfStackReserve,DllCharacteristics,ResourceSize,Benign
0,332,0,0,0,4,0,0,8192,8,0,3,1048576,34112,672,1
1,34404,84,121728,10,10,126576,4930,0,14,10,8,262144,16864,1024,1
2,332,0,0,0,4,0,0,8192,8,0,3,1048576,34112,672,1
3,34404,84,19904,10,10,21312,252,18160,14,10,6,262144,16736,1040,1
4,34404,84,97728,10,10,105792,1852,70592,14,10,7,262144,16736,1096,1


# Parâmetros dos Modelos


## GBM


In [ ]:
# ======== ESPAÇOS PARA AMOSTRAGEM ALEATÓRIA (GBM) ========
n_estimators_choices = list(range(50, 501, 10))
max_depth_choices = list(range(2, 50, 2))
min_samples_split_choices = list(range(2, 21))
min_samples_leaf_choices = list(range(1, 21))

# ================== LOOP PRINCIPAL ==================
rows = []

for seed in SEEDS:
    random.seed(int(seed))
    np.random.seed(seed)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for i in range(10):  # 25 testes por seed
        params_rf = {
            "n_estimators": random.choice(n_estimators_choices),
            "max_depth": random.choice(max_depth_choices),
            "min_samples_split": random.choice(min_samples_split_choices),
            "min_samples_leaf": random.choice(min_samples_leaf_choices),
            "random_state": seed,
        }
        gbm = GradientBoostingClassifier(**params_rf)

        # Cross Validation 5-fold
        scores_rf = cross_val_score(
            gbm, X, Y,
            cv=skf,
            scoring="accuracy",
        )

        rows.append({
            "seed": seed,
            "iter": i + 1,
            "clf__n_estimators": params_rf["n_estimators"],
            "clf__max_depth": params_rf["max_depth"],
            "clf__min_samples_split": params_rf["min_samples_split"],
            "clf__min_samples_leaf": params_rf["min_samples_leaf"],
            "mean_accuracy": float(np.mean(scores_rf)),
            "std_accuracy": float(np.std(scores_rf)),
        })

        # ================== SALVAR RESULTADOS ==================
        pd.DataFrame(rows).to_csv("../resultados/parametros_gbm_ransomware.csv", index=False, decimal=".", sep=";")

best_gbm = pd.DataFrame(rows).loc[df["mean_accuracy"].idxmax()]
print("Melhores Parâmetros GBM", best_gbm)



## RF

In [ ]:
# ======== ESPAÇOS PARA AMOSTRAGEM ALEATÓRIA ========
n_estimators_choices = list(range(50, 501, 10))
max_depth_choices = list(range(2, 101, 2))
min_samples_split_choices = list(range(2, 21))
min_samples_leaf_choices = list(range(1, 21))

# ======== LOOP PRINCIPAL ========
rows = []

for seed in SEEDS:
    random.seed(int(seed))
    np.random.seed(seed)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for i in range(10):  # 25 testes por seed
        params_rf = {
            "n_estimators": random.choice(n_estimators_choices),
            "max_depth": random.choice(max_depth_choices),
            "min_samples_split": random.choice(min_samples_split_choices),
            "min_samples_leaf": random.choice(min_samples_leaf_choices),
            "random_state": seed,
        }

        rf = RandomForestClassifier(**params_rf)

        # Cross Validation 5-fold
        scores_rf = cross_val_score(
            rf, X, Y,
            cv=skf,
            scoring="accuracy",
        )

        rows.append({
            "seed": seed,
            "iter": i + 1,
            "clf__n_estimators": params_rf["n_estimators"],
            "clf__max_depth": params_rf["max_depth"],
            "clf__min_samples_split": params_rf["min_samples_split"],
            "clf__min_samples_leaf": params_rf["min_samples_leaf"],
            "mean_accuracy": float(np.mean(scores_rf)),
            "std_accuracy": float(np.std(scores_rf)),
        })

        # ======== SALVAR RESULTADOS ========
        pd.DataFrame(rows).to_csv("../resultados/parametros_rf_ransomware.csv", index=False, decimal=".", sep=";")

best_rf = pd.DataFrame(rows).loc[df["mean_accuracy"].idxmax()]
print("Melhores Parâmetros RF", best_rf)


## SVM

In [ ]:
# ======== ESPAÇOS PARA AMOSTRAGEM ALEATÓRIA (SVM) ========
kernel_choices = ["linear", "rbf", "poly", "sigmoid"]
C_choices = list(range(50, 501, 10))
gamma_choices = list(np.arange(0, 1, 0.05))

# ================== LOOP PRINCIPAL ==================
rows = []

for seed in SEEDS:
    random.seed(int(seed))
    np.random.seed(seed)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    for i in range(25):  # 25 testes por seed
        kernel = random.choice(kernel_choices)
        C = random.choice(C_choices)
        gamma = random.choice(gamma_choices)

        svm = SVC(kernel=kernel, C=C, gamma=gamma, random_state=seed)

        # Cross Validation 5-fold
        scores_rf = cross_val_score(
            svm, X, Y,
            cv=skf,
            scoring="accuracy",
        )

        # monta linha de saída
        row = {
            "seed": seed,
            "iter": i + 1,
            "clf__kernel": kernel,
            "clf__C": C,
            "clf__gamma": gamma,
            "mean_accuracy": float(np.mean(scores_rf)),
            "std_accuracy": float(np.std(scores_rf)),
        }

        # ================== SALVAR RESULTADOS ==================
        pd.DataFrame(rows).to_csv("../resultados/parametros_svm_ransomware.csv", index=False, decimal=".", sep=";")

best_svm = pd.DataFrame(rows).loc[df["mean_accuracy"].idxmax()]
print("Melhores Parâmetros SVM", best_svm)

Seed 1000 - Iteração 1
Seed 1000 - Iteração 2
Seed 1000 - Iteração 3
Seed 1000 - Iteração 4
Seed 1000 - Iteração 5
Seed 1000 - Iteração 6
Seed 1000 - Iteração 7
Seed 1000 - Iteração 8


# Execução dos Testes

In [2]:
parametros_gbm = pd.read_csv("../resultados/parametros_gbm_ransomware.csv", sep=";", decimal=".")
parametros_rf = pd.read_csv("../resultados/parametros_rf_ransomware.csv", sep=";", decimal=".")

best_gbm = parametros_gbm.sort_values(
    by=["mean_accuracy", "std_accuracy"], ascending=[False, True]
).head(1).iloc[0]

best_rf = parametros_rf.sort_values(
    by=["mean_accuracy", "std_accuracy"], ascending=[False, True]
).head(1).iloc[0]

best_rf


seed                      8000.000000
iter                         6.000000
clf__n_estimators          450.000000
clf__max_depth              76.000000
clf__min_samples_split       3.000000
clf__min_samples_leaf        1.000000
mean_accuracy                0.996527
std_accuracy                 0.000505
Name: 75, dtype: float64

## Validação Cruzada


In [3]:
# ================== PARÂMETROS ==================
params_gbm = {
    "n_estimators": int(best_gbm["clf__n_estimators"]),
    "max_depth": int(best_gbm["clf__max_depth"]),
    "min_samples_split": int(best_gbm["clf__min_samples_split"]),
    "min_samples_leaf": int(best_gbm["clf__min_samples_leaf"]),
}

params_rf = {
    "n_estimators": int(best_rf["clf__n_estimators"]),
    "max_depth": int(best_rf["clf__max_depth"]),
    "min_samples_split": int(best_rf["clf__min_samples_split"]),
    "min_samples_leaf": int(best_rf["clf__min_samples_leaf"]),
}

# ================== RESULTADOS ==================
results_gbm = []
results_rf = []

for seed in SEEDS:
    random.seed(int(seed))
    np.random.seed(seed)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    # ===== RF =====
    model_rf = RandomForestClassifier(**params_rf, random_state=seed)
    scores_rf = cross_val_score(model_rf, X, Y, cv=skf, scoring="accuracy")
    results_rf.append({
        "seed": seed,
        "mean_accuracy": scores_rf.mean(),
        "std_accuracy": scores_rf.std(),
        "fold_1": scores_rf[0],
        "fold_2": scores_rf[1],
        "fold_3": scores_rf[2],
        "fold_4": scores_rf[3],
        "fold_5": scores_rf[4],
    })
    # ===== GBM =====
    model_gbm = GradientBoostingClassifier(**params_gbm, random_state=seed)
    scores_gbm = cross_val_score(model_gbm, X, Y, cv=skf, scoring="accuracy")
    results_gbm.append({
        "seed": seed,
        "mean_accuracy": scores_gbm.mean(),
        "std_accuracy": scores_gbm.std(),
        "fold_1": scores_gbm[0],
        "fold_2": scores_gbm[1],
        "fold_3": scores_gbm[2],
        "fold_4": scores_gbm[3],
        "fold_5": scores_gbm[4],
    })

    # salvar novo csv
    pd.DataFrame(results_gbm).to_csv("../resultados/resultados_validacao_cruzada_gbm_ransomware.csv", index=False, decimal=".",
                                    sep=";")
    pd.DataFrame(results_rf).to_csv("../resultados/resultados_validacao_cruzada_rf_ransomware.csv", index=False, decimal=".",
                                    sep=";")



[0.99711931 0.99647915 0.99679923 0.99727935 0.99607906]
[0.99719933 0.99647915 0.99639914 0.9962391  0.99703929]
[0.99695927 0.99687925 0.99663919 0.99647915 0.99631912]
[0.99591902 0.99647915 0.99703929 0.99703929 0.99711931]
[0.99639914 0.99687925 0.99719933 0.99663919 0.99687925]
[0.99703929 0.99607906 0.99687925 0.99735937 0.99615908]
[0.99551892 0.99719933 0.99679923 0.99735937 0.99671921]
[0.99663919 0.99615908 0.99575898 0.99695927 0.99711931]
[0.99703929 0.99663919 0.99679923 0.9975194  0.99687925]
[0.99743939 0.99735937 0.99703929 0.99631912 0.99575898]


## Holdout

In [4]:
results_gbm = []
results_rf = []

for seed in SEEDS:
    random.seed(int(seed))
    np.random.seed(seed)

    # Hold-Out
    X_train, X_test, y_train, y_test = train_test_split(
        X, Y,
        test_size=0.4,
        shuffle=True,
        random_state=seed,
    )

    # ===== RF =====
    model_rf = RandomForestClassifier(
        **params_rf,
        random_state=seed
    )
    model_rf.fit(X_train, y_train)
    acc_rf = model_rf.score(X_test, y_test)

    results_rf.append({
        "seed": seed,
        "accuracy": acc_rf
    })

    # ===== GBM =====
    model_gbm = GradientBoostingClassifier(
        **params_gbm,
        random_state=seed
    )
    model_gbm.fit(X_train, y_train)
    acc_gbm = model_gbm.score(X_test, y_test)

    results_gbm.append({
        "seed": seed,
        "accuracy": acc_gbm
    })

    # salvar novo csv
    pd.DataFrame(results_gbm).to_csv("../resultados/resultados_holdout_gbm_ransomware.csv", index=False, sep=";", decimal=".")
    pd.DataFrame(results_rf).to_csv("../resultados/resultados_holdout_rf_ransomware.csv", index=False, sep=";", decimal=".")


